# BDIFF — exploration des exports officiels 2024 et 2025

**Questions : Q1, Q2 et Q4.** Nous étudions d'abord la Haute-Garonne (31), puis les 13 départements d'Occitanie comme comparateurs.

On utilise les deux ZIP téléchargés depuis le portail BDIFF. Ils contiennent les incendies du **type F** et **deux lignes de préambule** avant l'en-tête CSV. Les archives sont conservées intactes.

**Réserve :** le runner GitHub Actions n'a pas vérifié le certificat du portail. Le contenu et les filtres ont été contrôlés ; la provenance cryptographique reste à confirmer avec un téléchargement indépendant.

**Relectures proposées :** Maeva (source), Chantal (qualité et licence), Stéphane (pipeline), Ludovic (indicateurs).

## 1. Charger les deux fichiers et vérifier les empreintes

Le manifeste conserve les URL, SHA-256, filtres, dates et observations sur les ZIP. Nous n'extrayons rien définitivement : les CSV sont lus en mémoire.

In [ ]:
from pathlib import Path
import csv, hashlib, io, json, zipfile
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "datasets/public/bdiff_officiel/manifest.json").exists():
    ROOT = ROOT.parent
DOSSIER = ROOT / "datasets/public/bdiff_officiel"
manifest = json.loads((DOSSIER / "manifest.json").read_text(encoding="utf-8"))
parts = []
for ref in manifest["extractions"]:
    fichier = DOSSIER / ref["fichier"]
    digest = hashlib.sha256(fichier.read_bytes()).hexdigest()
    assert digest == ref["sha256"], f"Empreinte différente : {fichier.name}"
    with zipfile.ZipFile(fichier) as archive:
        brut = archive.read("Incendies.csv").decode("utf-8-sig")
    lignes = brut.splitlines()
    assert lignes[2].startswith("Année;Numéro;Département;")
    df_annee = pd.read_csv(io.StringIO("\n".join(lignes[2:])), sep=";", dtype=str)
    df_annee["source_annee"] = str(ref["annee"])
    parts.append(df_annee)
    print(ref["annee"], len(df_annee), "événements", "SHA-256 :", digest)
df = pd.concat(parts, ignore_index=True)
display(df.head())

## 2. Contrôler la qualité au vrai grain événement

L'identifiant BDIFF `Numéro` doit être examiné avant de choisir une clé définitive. Les codes département et INSEE restent des chaînes, pour préserver les zéros initiaux.

In [ ]:
print("Colonnes :", list(df.columns))
display(pd.DataFrame({
    "manquants": df.isna().sum(),
    "pct_manquants": (100 * df.isna().mean()).round(2),
    "distincts": df.nunique(dropna=True)
}).sort_values("pct_manquants", ascending=False))
print("Doublons Année + Numéro :", df.duplicated(["Année", "Numéro"]).sum())
print("Années observées :", sorted(df["Année"].unique()))
display(pd.crosstab(df["Année"], df["Département"]))

## 3. Haute-Garonne et départements comparables

Des effectifs très faibles dans le 31 peuvent fragiliser une règle de seuil : les comparaisons régionales ne sont pas là pour remplacer notre commanditaire, mais pour tester la robustesse de nos observations.

In [ ]:
import matplotlib.pyplot as plt
par_departement = df.groupby(["Année", "Département"]).size().unstack(fill_value=0)
display(par_departement)
ax = par_departement.T.plot(kind="bar", figsize=(12, 5), title="Incendies BDIFF de type F par département d'Occitanie")
ax.set_ylabel("Nombre d'événements")
plt.tight_layout()
plt.show()
print("Feux recensés dans le 31 :", df.loc[df["Département"].eq("31")].groupby("Année").size().to_dict())

## 4. Raccord au journal des contrôles exécutés

Le pipeline fournit un rapprochement **technique** avec les archives météo, mais il ne prouve pas encore que la prévision J+1/J+2 correspond au jour de l'incendie.

In [ ]:
rapport = ROOT / "docs/resultats/profilage_bdiff_officiel_v1.json"
if rapport.exists():
    constat = json.loads(rapport.read_text(encoding="utf-8"))
    display(pd.DataFrame(constat["raccords_techniques_NON_KPI"]))
    print("Limites :")
    for texte in constat["limites"]:
        print("•", texte)
else:
    print("Rapport absent : lancer python3 scripts/profiler_bdiff_original.py")

## 5. Conclusions de travail

- Le téléchargement a rapporté 413 incendies de type F en Occitanie pour 2024 et 472 pour 2025.
- La Haute-Garonne compte **très peu d'événements** dans ces exports : toute recommandation locale devra tenir compte de cette incertitude.
- Q1 demandera un dénominateur IGN (surface forestière) et une harmonisation INSEE.
- Q2 et Q3 demandent la validation de la date de validité de **niveau_j1 / niveau_j2** ; la simple jointure sur la date est insuffisante.
- Il reste à corroborer l'authenticité du téléchargement TLS depuis un navigateur vérifiant les certificats, puis à documenter les éventuels écarts.

**Relecture attendue :** Maeva, Chantal, Stéphane et Ludovic ; aucun retour n'est attribué sans leur intervention.